In [ ]:
# @title 1. Install & Update Libraries
!pip install -q -U "transformers>=4.41.2" accelerate bitsandbytes langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers pdfplumber
!pip install -q sentence-transformers
print("-" * 50)
print("✅ LIBRARIES INSTALLED")

--------------------------------------------------
✅ LIBRARIES INSTALLED


In [ ]:
!pip install -q gradio

In [ ]:
!pip install rouge-score bert-score nltk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.9 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=89d61596a6f9ac020f09203718236d8f22c63010d4102d33b1387f503b2e4583
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
import os, csv, time, torch, gc, pdfplumber, warnings, re, string, collections
import pandas as pd
from datetime import datetime
from typing import List
from huggingface_hub import login
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import CrossEncoder, SentenceTransformer, util

# --- NEW METRIC LIBRARIES ---
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score

# --- AUTHENTICATION ---
HF_TOKEN = ""
login(token=HF_TOKEN)
warnings.filterwarnings("ignore")

# ==========================================
# 1. ENHANCED PDF LOADER
# ==========================================
def load_pdf_smart(path):
    print(f"📂 Analyzing {path}...")
    documents = []
    try:
        with pdfplumber.open(path) as pdf:
            for i, page in enumerate(pdf.pages):
                text = page.extract_text() or ""
                tables = page.extract_tables()
                table_data = ""
                if tables:
                    for table in tables:
                        table_data += "\n" + "\n".join(["| " + " | ".join([str(c).replace('\n', ' ') if c else "" for c in row]) + " |" for row in table])
                full_content = f"{text}\n{table_data}"
                documents.append(Document(
                    page_content=full_content,
                    metadata={"page": i+1, "source": os.path.basename(path)}
                ))
        return documents
    except Exception as e:
        print(f"❌ Error loading PDF: {e}")
        return []

# ==========================================
# 2. MEDICAL EMBEDDINGS (MedCPT)
# ==========================================
class MedCPTEmbeddings(Embeddings):
    def __init__(self, load_article_encoder=True):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.models = {
            "qry_tok": AutoTokenizer.from_pretrained("ncbi/MedCPT-Query-Encoder"),
            "qry_mod": AutoModel.from_pretrained("ncbi/MedCPT-Query-Encoder")
        }
        if load_article_encoder:
            self.models["art_tok"] = AutoTokenizer.from_pretrained("ncbi/MedCPT-Article-Encoder")
            self.models["art_mod"] = AutoModel.from_pretrained("ncbi/MedCPT-Article-Encoder").to(self.device)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        all_embeddings = []
        batch_size = 8
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            inputs = self.models["art_tok"](batch, max_length=512, padding=True, truncation=True, return_tensors="pt").to(self.device)
            with torch.no_grad():
                output = self.models["art_mod"](**inputs)
                all_embeddings.extend(output.last_hidden_state[:, 0, :].cpu().tolist())
        return all_embeddings

    def embed_query(self, text: str) -> List[float]:
        self.models["qry_mod"].to(self.device)
        inputs = self.models["qry_tok"]([text], max_length=512, padding=True, truncation=True, return_tensors="pt").to(self.device)
        with torch.no_grad():
            output = self.models["qry_mod"](**inputs)
            result = output.last_hidden_state[:, 0, :][0].cpu().tolist()
        self.models["qry_mod"].to("cpu")
        del inputs, output
        torch.cuda.empty_cache()
        return result

    def unload_article_encoder(self):
        if "art_mod" in self.models:
            del self.models["art_mod"], self.models["art_tok"]
            torch.cuda.empty_cache()
            gc.collect()

# ==========================================
# 3. INGESTION & VECTOR STORE
# ==========================================
INDEX_PATH = "faiss_esc_index"
PDF_PATH = "/content/2024ESC-compressed.pdf"

if not os.path.exists(INDEX_PATH):
    print("⚙️ Building Vector Index...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150, separators=["\n\n", "\n|", "\n", " "])
    raw_docs = load_pdf_smart(PDF_PATH)
    if not raw_docs: raise ValueError("PDF not found or empty!")
    splits = text_splitter.split_documents(raw_docs)
    medcpt = MedCPTEmbeddings(load_article_encoder=True)
    vectorstore = FAISS.from_documents(splits, medcpt)
    vectorstore.save_local(INDEX_PATH)
    medcpt.unload_article_encoder()
else:
    print("✅ Loading existing Vector Index...")
    medcpt = MedCPTEmbeddings(load_article_encoder=False)

vectorstore = FAISS.load_local(INDEX_PATH, embeddings=medcpt, allow_dangerous_deserialization=True)

# ==========================================
# 4. INFERENCE PIPELINE
# ==========================================
print("⚙️ Loading Reranker & LLM...")
reranker = CrossEncoder('BAAI/bge-reranker-base', device='cpu')

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)
terminators = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")]

print("⚙️ Loading Semantic Similarity Model...")
metric_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

# ==========================================
# 5. METRICS EVALUATION FUNCTIONS
# ==========================================
def normalize_text(s):
    """Standardizes text for exact match and F1 calculations."""
    if pd.isna(s): return ""
    s = str(s).lower()
    s = s.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(s.split())

def calc_accuracy_metrics(pred, truth):
    norm_pred = normalize_text(pred).split()
    norm_truth = normalize_text(truth).split()

    # 1. Exact Match
    exact_match = int(norm_pred == norm_truth)

    # 2. Token F1/Precision/Recall
    common = collections.Counter(norm_pred) & collections.Counter(norm_truth)
    num_same = sum(common.values())

    if len(norm_pred) == 0 or len(norm_truth) == 0:
        f1 = precision = recall = int(norm_pred == norm_truth)
    elif num_same == 0:
        f1 = precision = recall = 0.0
    else:
        precision = num_same / len(norm_pred)
        recall = num_same / len(norm_truth)
        f1 = (2 * precision * recall) / (precision + recall)

    return exact_match, round(precision, 4), round(recall, 4), round(f1, 4)

def calc_lexical_metrics(pred, truth):
    if pd.isna(truth) or not truth: return 0, 0, 0, 0, 0, 0
    # 3. BLEU
    smoothie = SmoothingFunction().method4
    ref = [normalize_text(truth).split()]
    hyp = normalize_text(pred).split()
    b1 = sentence_bleu(ref, hyp, weights=(1, 0, 0, 0), smoothing_function=smoothie)
    b2 = sentence_bleu(ref, hyp, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)
    b4 = sentence_bleu(ref, hyp, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)

    # 4. ROUGE
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(str(truth), str(pred))

    return round(b1,4), round(b2,4), round(b4,4), \
           round(rouge_scores['rouge1'].fmeasure,4), \
           round(rouge_scores['rouge2'].fmeasure,4), \
           round(rouge_scores['rougeL'].fmeasure,4)

def calc_semantic_metrics(pred, truth):
    if pd.isna(truth) or not truth: return 0,0,0,0

    # 5. BERTScore (Loaded on CPU to save VRAM)
    P, R, F1 = bert_score([str(pred)], [str(truth)], lang="en", model_type="distilbert-base-uncased", device="cpu")

    # 6. Cosine Semantic Similarity
    embeddings = metric_model.encode([str(pred), str(truth)], convert_to_tensor=True)
    cosine_sim = float(util.pytorch_cos_sim(embeddings[0], embeddings[1])[0][0])

    return round(P.item(),4), round(R.item(),4), round(F1.item(),4), round(cosine_sim,4)

def calc_rag_metrics_llm(query, context, response, expected):
    """
    7, 8, 9. LLM-as-a-judge for Faithfulness, Answer Relevancy, and Context Recall.
    """
    eval_prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert RAG evaluator. Grade the response based on the following three metrics.
Output ONLY three comma-separated numbers between 0 and 10 (e.g., "9, 8, 10"). Do not output any other text.
1. Faithfulness: Is the Response completely grounded in the Context? (10=fully grounded, 0=hallucinated)
2. Answer Relevancy: Does the Response address the Query? (10=perfectly answers, 0=irrelevant)
3. Context Recall: Is the Expected Answer found within the Context? (10=fully found, 0=missing)<|eot_id|><|start_header_id|>user<|end_header_id|>
Query: {query}
Expected Answer: {expected}
Context: {context}
Response: {response}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

    inputs = tokenizer(eval_prompt, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=10, temperature=0.01)
    score_text = tokenizer.decode(output[0], skip_special_tokens=True).split("assistant")[-1].strip()

    del inputs, output
    torch.cuda.empty_cache()

    # Parse LLM output
    try:
        scores = [int(re.search(r'\d+', s).group()) for s in score_text.split(',')]
        if len(scores) >= 3:
            return scores[0], scores[1], scores[2]
    except Exception as e:
        pass
    return 0, 0, 0

# ==========================================
# 6. CORE GENERATION FUNCTION
# ==========================================
def get_answer(query, expected):
    # Retrieve & Rerank
    initial_docs = vectorstore.similarity_search(query, k=15)
    reranker.model.to("cuda")
    scores = reranker.predict([[query, d.page_content] for d in initial_docs])
    reranker.model.to("cpu")
    torch.cuda.empty_cache()

    top_results = sorted(zip(initial_docs, scores), key=lambda x: x[1], reverse=True)[:5]
    top_docs, top_scores = zip(*top_results)

    context = ""
    top_pages = []
    for d in top_docs:
        p = str(d.metadata.get('page', '??'))
        if p not in top_pages: top_pages.append(p)
        context += f"[Page {p}]\n{d.page_content}\n\n"

    # Generation Prompt
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a Cardiology Assistant. Answer based ONLY on the context. Be concise. Cite Page Numbers.<|eot_id|><|start_header_id|>user<|end_header_id|>
Context: {context}
Question: {query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

    # 10. System Metrics Tracking
    gen_start = time.time()
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=300, temperature=0.1)

    # Latency & Token calculation
    gen_time = time.time() - gen_start
    generated_tokens = len(output[0]) - len(inputs['input_ids'][0])
    tokens_per_sec = generated_tokens / gen_time if gen_time > 0 else 0

    response = tokenizer.decode(output[0], skip_special_tokens=True).split("assistant")[-1].strip()
    del inputs, output
    torch.cuda.empty_cache()

    # Calculate all Evaluation Metrics
    exact_match, tok_p, tok_r, tok_f1 = calc_accuracy_metrics(response, expected)
    bleu1, bleu2, bleu4, rouge1, rouge2, rougeL = calc_lexical_metrics(response, expected)
    bert_p, bert_r, bert_f1, cos_sim = calc_semantic_metrics(response, expected)
    faithfulness, answer_rel, ctx_recall = calc_rag_metrics_llm(query, context, response, expected)

    return {
        "Response": response,
        "Pages": ", ".join(top_pages),
        "Exact Match": exact_match,
        "Token F1": tok_f1, "Token P": tok_p, "Token R": tok_r,
        "BLEU-1": bleu1, "BLEU-2": bleu2, "BLEU-4": bleu4,
        "ROUGE-1": rouge1, "ROUGE-2": rouge2, "ROUGE-L": rougeL,
        "BERTScore F1": bert_f1, "BERTScore P": bert_p, "BERTScore R": bert_r,
        "Semantic Similarity": cos_sim,
        "Faithfulness": faithfulness,
        "Answer Relevancy": answer_rel,
        "Context Recall": ctx_recall,
        "Latency (s)": round(gen_time, 2),
        "Tokens/sec": round(tokens_per_sec, 2)
    }

# ==========================================
# 7. AUTOMATED TEST SUITE WITH FULL METRICS
# ==========================================
input_csv_path = "RAG Test Prompts for Medical Guidelines - RAG Test Prompts for Medical Guidelines.csv"
output_csv_path = "rag_evaluation_automated_v5_full_metrics.csv"

try:
    print(f"📂 Loading prompts from {input_csv_path}...")
    df = pd.read_csv(input_csv_path)
except FileNotFoundError:
    print(f"❌ Error: Could not find {input_csv_path}.")
    df = pd.DataFrame()

if not df.empty:
    # Initialize CSV Headers with the 10 metric categories
    headers = [
        "Query", "Expected", "Response", "Pages",
        "Exact Match", "Token F1", "Token P", "Token R",
        "BLEU-1", "BLEU-2", "BLEU-4", "ROUGE-1", "ROUGE-2", "ROUGE-L",
        "BERTScore F1", "BERTScore P", "BERTScore R", "Semantic Similarity",
        "Faithfulness", "Answer Relevancy", "Context Recall",
        "Latency (s)", "Tokens/sec"
    ]

    with open(output_csv_path, mode='w', newline='', encoding='utf-8') as f:
        csv.writer(f).writerow(headers)

    print(f"\n🚀 Starting Full Metric Evaluation of {len(df)} questions...\n")

    for index, row in df.iterrows():
        q = row['Prompt (Question)']
        expected = row['Ground Truth (Expected Answer)']

        print(f"🔄 Processing Q{index+1}/{len(df)}...")
        res = get_answer(q, expected)

        print(f"✅ RAG Scores (Fth/Rel/Ctx): {res['Faithfulness']},{res['Answer Relevancy']},{res['Context Recall']} | F1: {res['Token F1']} | BERT_F1: {res['BERTScore F1']} | Speed: {res['Tokens/sec']} T/s")

        with open(output_csv_path, mode='a', newline='', encoding='utf-8') as f:
            csv.writer(f).writerow([
                q, expected, res['Response'], res['Pages'],
                res['Exact Match'], res['Token F1'], res['Token P'], res['Token R'],
                res['BLEU-1'], res['BLEU-2'], res['BLEU-4'], res['ROUGE-1'], res['ROUGE-2'], res['ROUGE-L'],
                res['BERTScore F1'], res['BERTScore P'], res['BERTScore R'], res['Semantic Similarity'],
                res['Faithfulness'], res['Answer Relevancy'], res['Context Recall'],
                res['Latency (s)'], res['Tokens/sec']
            ])

        gc.collect()
        torch.cuda.empty_cache()

    print(f"\n✨ DONE! Comprehensive Evaluation results saved to {output_csv_path}")

✅ Loading existing Vector Index...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

⚙️ Loading Reranker & LLM...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`torch_dtype` is deprecated! Use `dtype` instead!


...

✅ RAG Scores (Fth/Rel/Ctx): 0,0,0 | F1: 0.8077 | BERT_F1: 0.9085 | Speed: 3.85 T/s
🔄 Processing Q20/20...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


✅ RAG Scores (Fth/Rel/Ctx): 9,10,10 | F1: 0.5263 | BERT_F1: 0.8395 | Speed: 4.82 T/s

✨ DONE! Comprehensive Evaluation results saved to rag_evaluation_automated_v5_full_metrics.csv
